# Darcy equation in a fractured domain: exercise 3

In this tutorial we present how to solve a Darcy equation with [PyGeoN](https://github.com/compgeo-mox/pygeon).  The unknowns are the velocity $q$ and the pressure $p$.

Let us consider the last test case of the 2d benchmark study reported here [2d problems](https://www.sciencedirect.com/science/article/pii/S0309170817300143).

We present *step-by-step* how to create the grid, declare the problem data, and finally solve the problem.


Let us start by importing the modules for the computation.

This is the guided ("fill in the code") version of `ex3.ipynb` -- work through the cells in order, completing each `__TODO__`. Compare against `ex3.ipynb` once you're done, or if you get stuck.

In [ ]:
import numpy as np
import scipy.sparse as sps

import porepy as pp
import pygeon as pg

We import the fracture network geometry and the mixed-dimensional grid from the module, in particular we are interested to a bidimensional network referenced as `flow_benchmark_2d_case_4`.

In [ ]:
# Define a rectangular domain in terms of range in the two dimensions
x_lim, y_lim = (0, 600), (0, 600)
bounding_box = {"xmin": x_lim[0], "xmax": x_lim[1], "ymin": y_lim[0], "ymax": y_lim[1]}
domain = pp.Domain(bounding_box=bounding_box)

# load the coordinates of the fractures
coords = np.loadtxt("benchmark_2d_case_4.csv", delimiter=",")

# collect all the fractures
fractures = __TODO__

# Define a fracture network in 2d
network = pp.create_fracture_network(fractures, domain)

# Show graphically the domain and the fracture
network.plot()

We now generate the mixed-dimensional grid from the domain and fracture network.

In [ ]:
# Set overall target cell size and target cell size close to the fracture.
mesh_args = {"cell_size": 100.0}

# Generate a mixed-dimensional grid
mdg = pp.create_mdg("simplex", mesh_args, network)
pg.convert_from_pp(mdg)
mdg.compute_geometry()

print(mdg)

Let us export the grid to ParaView

In [ ]:
save = pp.Exporter(mdg, "grid", folder_name="ex3")
save.write_vtu()

Let us declare the finite element spaces that we are going to use

In [ ]:
key = "flow"

# declare the discretization objects, useful to setup the data
rt0 = pg.RT0(key)
p0 = pg.PwConstants(key)

We introduce now the data to solve a single-phase flow problem, in particular we assume that the fracture is highly-permeable compared to the surrounding porous media. For doing this, we loop on all the grid and the associated data of the grid bucket, in this case the matrix grid and the fracture grid. Here we proceed as before, by adding the specific properties of the problem.

In [ ]:
bc_val = []
bc_ess = []

# Fracture data
aperture = 1e-2
fracture_perm = 1e-8  # 1e-17 1e-8
matrix_perm = 1e-12

scalar_source = []
for sd, data in mdg.subdomains(return_data=True):
    if sd.dim == 1:
        # effective permeability for the fracture
        eff_fracture_perm = aperture * fracture_perm * np.ones(sd.num_cells)
        inv_perm = __TODO__
    else:
        # unitary permeability tensor for the rock matrix
        inv_perm = __TODO__

    param = {pg.SECOND_ORDER_TENSOR: inv_perm}
    pp.initialize_data(data, key, param)

    # with the following steps we identify the portions of the boundary
    # to impose the boundary conditions
    left = __TODO__
    right = __TODO__
    left_right = np.logical_or(left, right)

    bottom = __TODO__
    top = __TODO__
    bottom_top = np.logical_or(bottom, top)

    # compute the pressure boundary condition, which is a natural condition for the RT0 space
    def p_bc(x):
        return __TODO__

    bc_val.append(__TODO__)
    bc_ess.append(__TODO__)

ess_p = np.zeros(mdg.num_subdomain_cells(), dtype=bool)
bc_ess.append(ess_p)

We consider now the interface between the fracture and the porous media, we can set additional data that govern the flow exchange between them, which is the normal permeability.

In [ ]:
for mg, data in mdg.interfaces(return_data=True):
    if mg.dim == 1:
        eff_normal_fracture_perm = fracture_perm / (aperture / 2)
    else:
        eff_normal_fracture_perm = fracture_perm / (aperture**2 / 2)
    pp.initialize_data(data, key, {pg.NORMAL_DIFFUSIVITY: eff_normal_fracture_perm})

Once the data are assigned to the mixed-dimensional grid, we construct the matrices. In particular, the linear system associated with the equation is given as
$$
\left(
\begin{array}{cc} 
A & -B^\top\\
B & 0
\end{array}
\right)
\left(
\begin{array}{c} 
q\\ 
p
\end{array}
\right)
=\left(
\begin{array}{c} 
0\\ 
f
\end{array}
\right)
$$<br>
$q$ now collects all the flux degrees of freedom for the rock matrix and fracture, similarly $p$ collects all the pressure degrees of freedom for the rock matrix and fracture.

To construct the saddle-point problem, we rely on the `scipy.sparse` function `block_array`. Once the matrix is created, we also construct the right-hand side containing the source term.

In [ ]:
# construct the local matrices
A = __TODO__
B = __TODO__

# assemble the saddle point problem
spp = __TODO__

# get the degrees of freedom for each variable
dof_p, dof_q = B.shape
dofs = np.array([dof_q, dof_p])

# assemble the right-hand side
rhs = np.zeros(dofs.sum())
rhs[: dofs[0]] += np.hstack(bc_val)

We need to solve the linear system, PyGeoN provides a framework for that. Once the problem is solved, we extract the two solutions $q$ and $p$.

In [ ]:
# solve the problem
ls = pg.LinearSystem(spp, rhs)
ls.flag_ess_bc(np.hstack(bc_ess), np.zeros(dofs.sum()))
x = ls.solve()

# split the solution into the components
idx = np.cumsum(dofs[:-1])
q, p = np.split(x, idx)

Since the computed $q$ is one value per facet of the grid, for visualization purposes we project the flux in each cell center as vector. First, we need to access the corresponding degrees of freedoms for each domain.
We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# post process variables
dof_q_loc = np.zeros(2, dtype=int)
dof_p_loc = np.zeros(2, dtype=int)

for sd, data in mdg.subdomains(return_data=True):
    # select the current dofs
    dof_q_loc = dof_q_loc[1] + [0, sd.num_faces]
    dof_p_loc = dof_p_loc[1] + [0, sd.num_cells]

    # extract the local solutions
    q_loc = q[dof_q_loc[0] : dof_q_loc[1]]
    p_loc = p[dof_p_loc[0] : dof_p_loc[1]]

    # compute the solution over each cell
    proj_q = rt0.eval_at_cell_centers(sd)
    cell_q = __TODO__
    cell_p = __TODO__

    # save the variables to be exported
    pp.set_solution_values("cell_q", cell_q, data, 0)
    pp.set_solution_values("cell_p", cell_p, data, 0)

save = pp.Exporter(mdg, "sol", folder_name="ex3")
save.write_vtu(["cell_q", "cell_p"])

We verify that the computed solution matches the expected reference values.

In [ ]:
# Consistency check
assert np.isclose(np.linalg.norm(cell_p), 846938.0466798104)
assert np.isclose(np.linalg.norm(cell_q), 0)